# 01 Importing Libraries

In [2]:
# Import libraries
import pandas as pd
import numpy as np
import os

# 02 Import Data

In [4]:
path = r'C:\Users\isava\OneDrive\Documents\CareerFoundry\Data Immersion\PythonFundamentals\Instacart Basket Analysis'
path

'C:\\Users\\isava\\OneDrive\\Documents\\CareerFoundry\\Data Immersion\\PythonFundamentals\\Instacart Basket Analysis'

In [5]:
df = pd.read_pickle(os.path.join(path,'02 Data', 'Prepared Data','ords_prods_indicators.pkl'))

In [6]:
df_subset = df[:1000000]
df_subset.shape

(1000000, 19)

In [7]:
df.head()

,order_id,user_id,order_number,orders_day_of_week,order_time_of_day,days_since_users_prior_order,product_id,add_to_cart_order,reordered,_merge,product_name,aisle_id,department_id,prices,_prodMerge,price_range_loc,busiest_day,busiest_days,busiest_period_of_day
0,2539329,1,1,2,8,NaN,196,1,0,both,Soda,77,7,9.0,both,Mid-range product,Regularly busy,Regularly busy,Average orders
1,2539329,1,1,2,8,NaN,14084,2,0,both,Organic Unsweetened Vanilla Almond Milk,91,16,12.5,both,Mid-range product,Regularly busy,Regularly busy,Average orders
2,2539329,1,1,2,8,NaN,12427,3,0,both,Original Beef Jerky,23,19,4.4,both,Low-range product,Regularly busy,Regularly busy,Average orders
3,2539329,1,1,2,8,NaN,26088,4,0,both,Aged White Cheddar Popcorn,23,19,4.7,both,Low-range product,Regularly busy,Regularly busy,Average orders
4,2539329,1,1,2,8,NaN,26405,5,0,both,XL Pick-A-Size Paper Towel Rolls,54,17,1.0,both,Low-range product,Regularly busy,Regularly busy,Average orders


## Aggregate

Find the aggregated mean of the “order_number” column grouped by “department_id” 

In [10]:
df_subset.groupby('department_id').agg({'order_number': ['mean']})

,order_number
,mean
department_id,
1,14.800024
2,17.091743
3,17.913544
4,17.893092
5,15.214270
6,15.382135
7,17.694027
8,16.458105


In [11]:
df.groupby('department_id').agg({'order_number': ['mean']})

,order_number
,mean
department_id,
1,15.457838
2,17.277920
3,17.170395
4,17.811403
5,15.215751
6,16.439806
7,17.225802
8,15.340650


This is essentially the average number of orders per user for each department id. As expected the subset values are different since it is calculating the mean of only a sample. It does appear that the general trends are about the same but the magnitude can be quite different.   
  
Next we will create a loyalty flag using transform().

In [13]:
df['max_order'] = df.groupby(['user_id'])['order_number'].transform(np.max)
df.head()

C:\Users\isava\AppData\Local\Temp\ipykernel_5212\1992126907.py:1: FutureWarning: The provided callable <function max at 0x0000026A5251F560> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  df['max_order'] = df.groupby(['user_id'])['order_number'].transform(np.max)


,order_id,user_id,order_number,orders_day_of_week,order_time_of_day,days_since_users_prior_order,product_id,add_to_cart_order,reordered,_merge,product_name,aisle_id,department_id,prices,_prodMerge,price_range_loc,busiest_day,busiest_days,busiest_period_of_day,max_order
0,2539329,1,1,2,8,NaN,196,1,0,both,Soda,77,7,9.0,both,Mid-range product,Regularly busy,Regularly busy,Average orders,10
1,2539329,1,1,2,8,NaN,14084,2,0,both,Organic Unsweetened Vanilla Almond Milk,91,16,12.5,both,Mid-range product,Regularly busy,Regularly busy,Average orders,10
2,2539329,1,1,2,8,NaN,12427,3,0,both,Original Beef Jerky,23,19,4.4,both,Low-range product,Regularly busy,Regularly busy,Average orders,10
3,2539329,1,1,2,8,NaN,26088,4,0,both,Aged White Cheddar Popcorn,23,19,4.7,both,Low-range product,Regularly busy,Regularly busy,Average orders,10
4,2539329,1,1,2,8,NaN,26405,5,0,both,XL Pick-A-Size Paper Towel Rolls,54,17,1.0,both,Low-range product,Regularly busy,Regularly busy,Average orders,10


In [14]:
df.loc[df['max_order'] > 40, 'loyalty_flag'] = 'Loyal customer'
df.loc[(df['max_order'] <= 40) & (df['max_order'] > 10), 'loyalty_flag'] = 'Regular customer'
df.loc[df['max_order'] <= 10, 'loyalty_flag'] = 'New customer'

In [15]:
df['loyalty_flag'].value_counts(dropna = False)

loyalty_flag
Regular customer    15876776
Loyal customer      10284093
New customer         6243990
Name: count, dtype: int64

The marketing team at Instacart wants to know whether there’s a difference between the spending habits of the three types of customers you identified. Use the loyalty flag you created and check the basic statistics of the product prices for each loyalty category (Loyal Customer, Regular Customer, and New Customer). What you’re trying to determine is whether the prices of products purchased by loyal customers differ from those purchased by regular or new customers.

In [17]:
df.groupby('loyalty_flag').agg({'prices': ['mean', 'min','max']})

prices              
                       mean  min      max
loyalty_flag                             
Loyal customer    10.386336  1.0  99999.0
New customer      13.294670  1.0  99999.0
Regular customer  12.495717  1.0  99999.0

So it appears that no matter how loyal the customer is they will spend within the same range of money. But on average the loyal customer spends less money than the regular customer. One potential hypothesis for this is that loyal customers are more savvy and take advantage of sales more often than a new customer. 

The team now wants to target different types of spenders in their marketing campaigns. This can be achieved by looking at the prices of the items people are buying. Create a spending flag for each user based on the average price across all their orders using the following criteria:
If the mean of the prices of products purchased by a user is lower than 10, then flag them as a “Low spender.”
If the mean of the prices of products purchased by a user is higher than or equal to 10, then flag them as a “High spender.”

In [20]:
df['avg_spending'] = df.groupby(['user_id'])['prices'].transform(np.average)
df.head()

,order_id,user_id,order_number,orders_day_of_week,order_time_of_day,days_since_users_prior_order,product_id,add_to_cart_order,reordered,_merge,...,department_id,prices,_prodMerge,price_range_loc,busiest_day,busiest_days,busiest_period_of_day,max_order,loyalty_flag,avg_spending
0,2539329,1,1,2,8,NaN,196,1,0,both,...,7,9.0,both,Mid-range product,Regularly busy,Regularly busy,Average orders,10,New customer,6.367797
1,2539329,1,1,2,8,NaN,14084,2,0,both,...,16,12.5,both,Mid-range product,Regularly busy,Regularly busy,Average orders,10,New customer,6.367797
2,2539329,1,1,2,8,NaN,12427,3,0,both,...,19,4.4,both,Low-range product,Regularly busy,Regularly busy,Average orders,10,New customer,6.367797
3,2539329,1,1,2,8,NaN,26088,4,0,both,...,19,4.7,both,Low-range product,Regularly busy,Regularly busy,Average orders,10,New customer,6.367797
4,2539329,1,1,2,8,NaN,26405,5,0,both,...,17,1.0,both,Low-range product,Regularly busy,Regularly busy,Average orders,10,New customer,6.367797


In [21]:
df.loc[df['avg_spending'] >= 10, 'spending_flag'] = 'High spender'
df.loc[df['avg_spending'] < 10, 'spending_flag'] = 'Low spender'

In [22]:
df['spending_flag'].value_counts(dropna = False)

spending_flag
Low spender     31770629
High spender      634230
Name: count, dtype: int64

There are many more Low spenders than high spenders.

In order to send relevant notifications to users within the app (for instance, asking users if they want to buy the same item again), the Instacart team wants you to determine frequent versus non-frequent customers. Create an order frequency flag that marks the regularity of a user’s ordering behavior according to the median in the “days_since_prior_order” column. The criteria for the flag should be as follows:
If the median of “days_since_prior_order” is higher than 20, then the customer should be labeled a “Non-frequent customer.”
If the median is higher than 10 and lower than or equal to 20, then the customer should be labeled a “Regular customer.”
If the median is lower than or equal to 10, then the customer should be labeled a “Frequent customer.”

In [25]:
df['median_order_frequency'] = df.groupby(['user_id'])['days_since_users_prior_order'].transform(np.median)
df.head(100)

C:\Users\isava\AppData\Local\Temp\ipykernel_5212\2396650721.py:1: FutureWarning: The provided callable <function median at 0x0000026A5264ACA0> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  df['median_order_frequency'] = df.groupby(['user_id'])['days_since_users_prior_order'].transform(np.median)


,order_id,user_id,order_number,orders_day_of_week,order_time_of_day,days_since_users_prior_order,product_id,add_to_cart_order,reordered,_merge,...,_prodMerge,price_range_loc,busiest_day,busiest_days,busiest_period_of_day,max_order,loyalty_flag,avg_spending,spending_flag,median_order_frequency
0,2539329,1,1,2,8,NaN,196,1,0,both,...,both,Mid-range product,Regularly busy,Regularly busy,Average orders,10,New customer,6.367797,Low spender,20.5
1,2539329,1,1,2,8,NaN,14084,2,0,both,...,both,Mid-range product,Regularly busy,Regularly busy,Average orders,10,New customer,6.367797,Low spender,20.5
2,2539329,1,1,2,8,NaN,12427,3,0,both,...,both,Low-range product,Regularly busy,Regularly busy,Average orders,10,New customer,6.367797,Low spender,20.5
3,2539329,1,1,2,8,NaN,26088,4,0,both,...,both,Low-range product,Regularly busy,Regularly busy,Average orders,10,New customer,6.367797,Low spender,20.5
4,2539329,1,1,2,8,NaN,26405,5,0,both,...,both,Low-range product,Regularly busy,Regularly busy,Average orders,10,New customer,6.367797,Low spender,20.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,738281,2,4,2,10,8.0,21150,13,0,both,...,both,Mid-range product,Regularly busy,Regularly busy,Most orders,14,Regular customer,7.515897,Low spender,13.0
96,1673511,2,5,3,11,8.0,47144,1,0,both,...,both,Mid-range product,Regularly busy,Least busy days,Average orders,14,Regular customer,7.515897,Low spender,13.0
97,1673511,2,5,3,11,8.0,5322,2,0,both,...,both,Low-range product,Regularly busy,Least busy days,Average orders,14,Regular customer,7.515897,Low spender,13.0
98,1673511,2,5,3,11,8.0,17224,3,0,both,...,both,Low-range product,Regularly busy,Least busy days,Average orders,14,Regular customer,7.515897,Low spender,13.0


In [26]:
df.loc[df['median_order_frequency'] <= 10, 'order_frequency_flag'] = 'Non-Frequent customer'
df.loc[(df['median_order_frequency'] <= 20) & (df['median_order_frequency'] > 10), 'order_frequency_flag'] = 'Regular customer'
df.loc[df['median_order_frequency'] > 20, 'order_frequency_flag'] = 'Frequent customer'

In [27]:
df['order_frequency_flag'].value_counts(dropna = False)

order_frequency_flag
Frequent customer        21559853
Regular customer          7208564
Non-Frequent customer     3636437
NaN                             5
Name: count, dtype: int64

The 5 Nan values are the customers who have only ever made one order and no subsequent order. It appears that most customers are frequent customers. 

# 04 Export Data Frame

In [30]:
#df_merged_large.to_csv(os.path.join(path, '02 Data','Prepared Data', 'orders_products_combined.csv'))
df.to_pickle(os.path.join(path, '02 Data','Prepared Data', 'ords_prods_grouped_indicators.pkl'))